In [2]:
import json, zipfile, re
from pathlib import Path

base = Path("")
root = base 
examples_dir = root / "examples"
schema_dir = root / "schema"
ontology_dir = root / "ontology"
for d in [root, examples_dir, schema_dir, ontology_dir]:
    d.mkdir(parents=True, exist_ok=True)

ROOT = "http://nkust.edu.tw/mislab/cnc/"
ONTOLOGY_NS = ROOT + "ontology/sth#"
SCHEMA_BASE = ROOT + "schema/"


In [1]:
# Base examples
examples = {
    "nc-file.example.json": {
        "nc_file_id": "partA_op10_v3",
        "file_name": "partA_op10.nc",
        "program_name": "O1001",
        "machine_id": "cncA",
        "material": "Ti-6Al-4V",
        "tool_id": "toolD6",
        "revision": "r03"
    },
    "nc-block.example.json": {
        "nc_file_id": "partA_op10_v3",
        "block_id": "N120",
        "line_no": 120,
        "command": "N120 G01 X25.4 Y10.0 Z-2.0 F827.6",
        "spindle_speed_rpm": 3448.0,
        "feed_rate_mm_min": 827.6,
        "x_mm": 25.4,
        "y_mm": 10.0,
        "z_mm": -2.0,
        "segment_id": "seg_N120",
        "tool_id": "toolD6"
    },
    "sensing-window.example.json": {
        "window_id": "w000231",
        "nc_file_id": "partA_op10_v3",
        "machine_id": "cncA",
        "holder_id": "sth01",
        "tool_id": "toolD6",
        "start_time": "2026-04-13T10:00:00.100Z",
        "end_time": "2026-04-13T10:00:00.200Z",
        "start_block_id": "N120",
        "end_block_id": "N122",
        "segment_id": "seg_N120",
        "x_mm": 25.4,
        "y_mm": 10.0,
        "z_mm": -2.0
    },
    "multi-sensor-window.example.json": {
        "window_id": "w000231",
        "nc_file_id": "partA_op10_v3",
        "machine_id": "cncA",
        "holder_id": "sth01",
        "tool_id": "toolD6",
        "block_id": "N120",
        "segment_id": "seg_N120",
        "torque_nm": 1.28,
        "vibration_rms_g": 0.31,
        "strain_ue": 122.4,
        "temperature_c": 38.2,
        "spindle_current_a": 4.8,
        "dominant_frequency_hz": 780.0,
        "x_mm": 25.4,
        "y_mm": 10.0,
        "z_mm": -2.0
    },
    "diagnosis.example.json": {
        "window_id": "w000231",
        "machine_id": "cncA",
        "holder_id": "sth01",
        "chatter_state": True,
        "chatter_score": 0.93,
        "confidence": 0.94,
        "state_label": "chatter",
        "model_version": "lstm-chatter-v2.1"
    },
    "prediction.example.json": {
        "window_id": "w000231",
        "machine_id": "cncA",
        "holder_id": "sth01",
        "tool_condition": "severe_wear",
        "wear_score": 0.89,
        "rul_min": 6.5,
        "wear_mode": "flank",
        "confidence": 0.91,
        "model_version": "lstm-wear-v1.8"
    },
    # New sequence-level examples
    "multi-window-sequence.example.json": {
        "sequence_id": "seq_partA_op10_001",
        "nc_file_id": "partA_op10_v3",
        "machine_id": "cncA",
        "holder_id": "sth01",
        "tool_id": "toolD6",
        "windows": [
            {
                "window_id": "w000001",
                "start_time": "2026-04-13T10:00:00.100Z",
                "end_time": "2026-04-13T10:00:00.200Z",
                "block_id": "N120",
                "segment_id": "seg_N120",
                "torque_nm": 1.28,
                "vibration_rms_g": 0.31,
                "strain_ue": 122.4,
                "temperature_c": 38.2,
                "spindle_current_a": 4.8,
                "dominant_frequency_hz": 780.0,
                "x_mm": 25.4,
                "y_mm": 10.0,
                "z_mm": -2.0
            },
            {
                "window_id": "w000002",
                "start_time": "2026-04-13T10:00:00.200Z",
                "end_time": "2026-04-13T10:00:00.300Z",
                "block_id": "N121",
                "segment_id": "seg_N121",
                "torque_nm": 1.33,
                "vibration_rms_g": 0.34,
                "strain_ue": 125.0,
                "temperature_c": 38.5,
                "spindle_current_a": 4.9,
                "dominant_frequency_hz": 790.0,
                "x_mm": 28.0,
                "y_mm": 11.2,
                "z_mm": -2.0
            },
            {
                "window_id": "w000003",
                "start_time": "2026-04-13T10:00:00.300Z",
                "end_time": "2026-04-13T10:00:00.400Z",
                "block_id": "N122",
                "segment_id": "seg_N122",
                "torque_nm": 1.41,
                "vibration_rms_g": 0.39,
                "strain_ue": 130.2,
                "temperature_c": 39.1,
                "spindle_current_a": 5.0,
                "dominant_frequency_hz": 820.0,
                "x_mm": 30.6,
                "y_mm": 12.4,
                "z_mm": -2.0
            }
        ]
    },
    "diagnosis-sequence.example.json": {
        "sequence_id": "seq_partA_op10_001",
        "machine_id": "cncA",
        "holder_id": "sth01",
        "diagnosis_windows": [
            {
                "window_id": "w000001",
                "chatter_state": False,
                "chatter_score": 0.12,
                "confidence": 0.96,
                "state_label": "stable",
                "model_version": "lstm-chatter-v2.1"
            },
            {
                "window_id": "w000002",
                "chatter_state": False,
                "chatter_score": 0.28,
                "confidence": 0.94,
                "state_label": "transition",
                "model_version": "lstm-chatter-v2.1"
            },
            {
                "window_id": "w000003",
                "chatter_state": True,
                "chatter_score": 0.93,
                "confidence": 0.94,
                "state_label": "chatter",
                "model_version": "lstm-chatter-v2.1"
            }
        ]
    },
    "prediction-sequence.example.json": {
        "sequence_id": "seq_partA_op10_001",
        "machine_id": "cncA",
        "holder_id": "sth01",
        "prediction_windows": [
            {
                "window_id": "w000001",
                "tool_condition": "sharp",
                "wear_score": 0.12,
                "rul_min": 48.0,
                "wear_mode": "unknown",
                "confidence": 0.95,
                "model_version": "lstm-wear-v1.8"
            },
            {
                "window_id": "w000002",
                "tool_condition": "mild_wear",
                "wear_score": 0.38,
                "rul_min": 31.0,
                "wear_mode": "flank",
                "confidence": 0.92,
                "model_version": "lstm-wear-v1.8"
            },
            {
                "window_id": "w000003",
                "tool_condition": "severe_wear",
                "wear_score": 0.89,
                "rul_min": 6.5,
                "wear_mode": "flank",
                "confidence": 0.91,
                "model_version": "lstm-wear-v1.8"
            }
        ]
    }
}


In [4]:
# Base schemas + sequence schemas
schemas = {
    "nc-file.schema.json": {
        "$id": SCHEMA_BASE + "nc-file.schema.json",
        "title": "NC File",
        "type": "object",
        "required": ["nc_file_id", "file_name", "program_name", "machine_id", "material", "tool_id"],
        "properties": {
            "nc_file_id": {"type": "string"},
            "file_name": {"type": "string"},
            "program_name": {"type": "string"},
            "machine_id": {"type": "string"},
            "material": {"type": "string"},
            "tool_id": {"type": "string"},
            "revision": {"type": "string"}
        },
        "additionalProperties": False
    },
    "nc-block.schema.json": {
        "$id": SCHEMA_BASE + "nc-block.schema.json",
        "title": "NC Block",
        "type": "object",
        "required": ["nc_file_id", "block_id", "line_no", "command"],
        "properties": {
            "nc_file_id": {"type": "string"},
            "block_id": {"type": "string"},
            "line_no": {"type": "integer"},
            "command": {"type": "string"},
            "spindle_speed_rpm": {"type": "number"},
            "feed_rate_mm_min": {"type": "number"},
            "x_mm": {"type": "number"},
            "y_mm": {"type": "number"},
            "z_mm": {"type": "number"},
            "segment_id": {"type": "string"},
            "tool_id": {"type": "string"}
        },
        "additionalProperties": False
    },
    "sensing-window.schema.json": {
        "$id": SCHEMA_BASE + "sensing-window.schema.json",
        "title": "Sensing Window",
        "type": "object",
        "required": ["window_id", "nc_file_id", "machine_id", "tool_id"],
        "properties": {
            "window_id": {"type": "string"},
            "nc_file_id": {"type": "string"},
            "machine_id": {"type": "string"},
            "holder_id": {"type": "string"},
            "tool_id": {"type": "string"},
            "start_time": {"type": "string", "format": "date-time"},
            "end_time": {"type": "string", "format": "date-time"},
            "start_block_id": {"type": "string"},
            "end_block_id": {"type": "string"},
            "segment_id": {"type": "string"},
            "x_mm": {"type": "number"},
            "y_mm": {"type": "number"},
            "z_mm": {"type": "number"}
        },
        "additionalProperties": False
    },
    "multi-sensor-window.schema.json": {
        "$id": SCHEMA_BASE + "multi-sensor-window.schema.json",
        "title": "Multi-Sensor Window",
        "type": "object",
        "required": ["window_id", "nc_file_id", "machine_id", "tool_id"],
        "properties": {
            "window_id": {"type": "string"},
            "nc_file_id": {"type": "string"},
            "machine_id": {"type": "string"},
            "holder_id": {"type": "string"},
            "tool_id": {"type": "string"},
            "block_id": {"type": "string"},
            "segment_id": {"type": "string"},
            "torque_nm": {"type": "number"},
            "vibration_rms_g": {"type": "number"},
            "strain_ue": {"type": "number"},
            "temperature_c": {"type": "number"},
            "spindle_current_a": {"type": "number"},
            "dominant_frequency_hz": {"type": "number"},
            "x_mm": {"type": "number"},
            "y_mm": {"type": "number"},
            "z_mm": {"type": "number"},
            "start_time": {"type": "string", "format": "date-time"},
            "end_time": {"type": "string", "format": "date-time"}
        },
        "additionalProperties": False
    },
    "diagnosis.schema.json": {
        "$id": SCHEMA_BASE + "diagnosis.schema.json",
        "title": "Diagnosis",
        "type": "object",
        "required": ["window_id", "machine_id", "chatter_state", "chatter_score", "confidence", "state_label", "model_version"],
        "properties": {
            "window_id": {"type": "string"},
            "machine_id": {"type": "string"},
            "holder_id": {"type": "string"},
            "chatter_state": {"type": "boolean"},
            "chatter_score": {"type": "number", "minimum": 0, "maximum": 1},
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "state_label": {"type": "string", "enum": ["stable", "transition", "chatter"]},
            "model_version": {"type": "string"}
        },
        "additionalProperties": False
    },
    "prediction.schema.json": {
        "$id": SCHEMA_BASE + "prediction.schema.json",
        "title": "Prediction",
        "type": "object",
        "required": ["window_id", "machine_id", "tool_condition", "wear_score", "rul_min", "wear_mode", "confidence", "model_version"],
        "properties": {
            "window_id": {"type": "string"},
            "machine_id": {"type": "string"},
            "holder_id": {"type": "string"},
            "tool_condition": {"type": "string", "enum": ["sharp", "mild_wear", "severe_wear"]},
            "wear_score": {"type": "number", "minimum": 0, "maximum": 1},
            "rul_min": {"type": "number", "minimum": 0},
            "wear_mode": {"type": "string", "enum": ["flank", "crater", "chipping", "unknown"]},
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "model_version": {"type": "string"}
        },
        "additionalProperties": False
    },
    "multi-window-sequence.schema.json": {
        "$id": SCHEMA_BASE + "multi-window-sequence.schema.json",
        "title": "Multi-Window Sequence",
        "type": "object",
        "required": ["sequence_id", "nc_file_id", "machine_id", "tool_id", "windows"],
        "properties": {
            "sequence_id": {"type": "string"},
            "nc_file_id": {"type": "string"},
            "machine_id": {"type": "string"},
            "holder_id": {"type": "string"},
            "tool_id": {"type": "string"},
            "windows": {
                "type": "array",
                "items": {"$ref": SCHEMA_BASE + "multi-sensor-window.schema.json"}
            }
        },
        "additionalProperties": False
    },
    "diagnosis-sequence.schema.json": {
        "$id": SCHEMA_BASE + "diagnosis-sequence.schema.json",
        "title": "Diagnosis Sequence",
        "type": "object",
        "required": ["sequence_id", "machine_id", "diagnosis_windows"],
        "properties": {
            "sequence_id": {"type": "string"},
            "machine_id": {"type": "string"},
            "holder_id": {"type": "string"},
            "diagnosis_windows": {
                "type": "array",
                "items": {"$ref": SCHEMA_BASE + "diagnosis.schema.json"}
            }
        },
        "additionalProperties": False
    },
    "prediction-sequence.schema.json": {
        "$id": SCHEMA_BASE + "prediction-sequence.schema.json",
        "title": "Prediction Sequence",
        "type": "object",
        "required": ["sequence_id", "machine_id", "prediction_windows"],
        "properties": {
            "sequence_id": {"type": "string"},
            "machine_id": {"type": "string"},
            "holder_id": {"type": "string"},
            "prediction_windows": {
                "type": "array",
                "items": {"$ref": SCHEMA_BASE + "prediction.schema.json"}
            }
        },
        "additionalProperties": False
    }
}


| Schema pattern                         | TTL suggestion                             |
| -------------------------------------- | ------------------------------------------ |
| `*.schema.json` for a main object      | `owl:Class`                                |
| scalar numeric field                   | `owl:DatatypeProperty` with `xsd:double`   |
| scalar integer field                   | `owl:DatatypeProperty` with `xsd:integer`  |
| string field                           | `owl:DatatypeProperty` with `xsd:string`   |
| time field                             | `owl:DatatypeProperty` with `xsd:dateTime` |
| `_id` field referencing another object | `owl:ObjectProperty` candidate             |
| array of subobjects                    | `hasX` object property                     |
| enum states                            | controlled classes or controlled labels    |


In [5]:

ttl_text = f"""@prefix sth: <{ONTOLOGY_NS}> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

sth:StarterOntology rdf:type owl:Ontology ;
    rdfs:label "Starter CNC Smart Tool Holder Ontology with Sequence Extensions"@en ;
    rdfs:comment "Starter ontology derived from JSON examples and JSON schemas for NC files, NC blocks, sensing windows, multi-sensor windows, diagnosis, prediction, and multi-window sequences."@en .

### Classes
sth:NCFile rdf:type owl:Class .
sth:NCBlock rdf:type owl:Class .
sth:SensingWindow rdf:type owl:Class .
sth:MultiSensorWindow rdf:type owl:Class .
sth:Diagnosis rdf:type owl:Class .
sth:Prediction rdf:type owl:Class .
sth:ToolPathSegment rdf:type owl:Class .
sth:Signal rdf:type owl:Class .
sth:TorqueSignal rdf:type owl:Class ; rdfs:subClassOf sth:Signal .
sth:VibrationSignal rdf:type owl:Class ; rdfs:subClassOf sth:Signal .
sth:StrainSignal rdf:type owl:Class ; rdfs:subClassOf sth:Signal .
sth:TemperatureSignal rdf:type owl:Class ; rdfs:subClassOf sth:Signal .
sth:SpindleCurrentSignal rdf:type owl:Class ; rdfs:subClassOf sth:Signal .
sth:ToolCondition rdf:type owl:Class .
sth:SevereWear rdf:type owl:Class ; rdfs:subClassOf sth:ToolCondition .
sth:MildWear rdf:type owl:Class ; rdfs:subClassOf sth:ToolCondition .
sth:SharpTool rdf:type owl:Class ; rdfs:subClassOf sth:ToolCondition .
sth:ProcessState rdf:type owl:Class .
sth:ChatterState rdf:type owl:Class ; rdfs:subClassOf sth:ProcessState .

### Sequence classes
sth:WindowSequence rdf:type owl:Class .
sth:DiagnosisSequence rdf:type owl:Class .
sth:PredictionSequence rdf:type owl:Class .

### Object properties
sth:belongsToNCFile rdf:type owl:ObjectProperty .
sth:alignedWithNCBlock rdf:type owl:ObjectProperty .
sth:alignedWithToolPathSegment rdf:type owl:ObjectProperty .
sth:belongsToWindow rdf:type owl:ObjectProperty .
sth:diagnosesWindow rdf:type owl:ObjectProperty .
sth:predictsWindow rdf:type owl:ObjectProperty .
sth:indicatesToolCondition rdf:type owl:ObjectProperty .
sth:indicatesProcessState rdf:type owl:ObjectProperty .
sth:hasWindow rdf:type owl:ObjectProperty .
sth:hasDiagnosisWindow rdf:type owl:ObjectProperty .
sth:hasPredictionWindow rdf:type owl:ObjectProperty .
sth:nextWindow rdf:type owl:ObjectProperty .
sth:previousWindow rdf:type owl:ObjectProperty .

### Datatype properties
sth:hasName rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .
sth:hasSequenceIdentifier rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .
sth:hasFileName rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .
sth:hasProgramName rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .
sth:hasMachineId rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .
sth:hasMaterialName rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .
sth:hasToolIdentifier rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .
sth:hasRevision rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .
sth:hasLineNumber rdf:type owl:DatatypeProperty ; rdfs:range xsd:integer .
sth:hasCommandText rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .
sth:hasSpindleSpeed rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasFeedRate rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasStartTime rdf:type owl:DatatypeProperty ; rdfs:range xsd:dateTime .
sth:hasEndTime rdf:type owl:DatatypeProperty ; rdfs:range xsd:dateTime .
sth:hasXCoordinate rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasYCoordinate rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasZCoordinate rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasTorqueValue rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasVibrationRMS rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasStrainValue rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasTemperatureValue rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasSpindleCurrentValue rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasDominantFrequency rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasScore rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasConfidence rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasStateLabel rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .
sth:hasModelVersion rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .
sth:hasWearScore rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasRUL rdf:type owl:DatatypeProperty ; rdfs:range xsd:double .
sth:hasWearMode rdf:type owl:DatatypeProperty ; rdfs:range xsd:string .

### Example individuals
sth:NCFile_partA_op10_v3 rdf:type sth:NCFile ;
    sth:hasName "partA_op10_v3" ;
    sth:hasFileName "partA_op10.nc" ;
    sth:hasProgramName "O1001" ;
    sth:hasMachineId "cncA" ;
    sth:hasMaterialName "Ti-6Al-4V" ;
    sth:hasToolIdentifier "toolD6" ;
    sth:hasRevision "r03" .

sth:NCBlock_N120 rdf:type sth:NCBlock ;
    sth:hasName "N120" ;
    sth:hasLineNumber 120 ;
    sth:hasCommandText "N120 G01 X25.4 Y10.0 Z-2.0 F827.6" ;
    sth:hasSpindleSpeed 3448.0 ;
    sth:hasFeedRate 827.6 ;
    sth:hasXCoordinate 25.4 ;
    sth:hasYCoordinate 10.0 ;
    sth:hasZCoordinate -2.0 ;
    sth:hasToolIdentifier "toolD6" ;
    sth:belongsToNCFile sth:NCFile_partA_op10_v3 .

sth:Seq_seq_partA_op10_001 rdf:type sth:WindowSequence ;
    sth:hasSequenceIdentifier "seq_partA_op10_001" ;
    sth:hasWindow sth:Window_w000001, sth:Window_w000002, sth:Window_w000003 .

sth:Window_w000001 rdf:type sth:MultiSensorWindow ;
    sth:hasName "w000001" ;
    sth:hasMachineId "cncA" ;
    sth:hasToolIdentifier "toolD6" ;
    sth:belongsToNCFile sth:NCFile_partA_op10_v3 ;
    sth:alignedWithNCBlock sth:NCBlock_N120 ;
    sth:hasTorqueValue 1.28 ;
    sth:hasVibrationRMS 0.31 .

sth:Window_w000002 rdf:type sth:MultiSensorWindow ;
    sth:hasName "w000002" ;
    sth:hasMachineId "cncA" ;
    sth:hasToolIdentifier "toolD6" ;
    sth:belongsToNCFile sth:NCFile_partA_op10_v3 .

sth:Window_w000003 rdf:type sth:MultiSensorWindow ;
    sth:hasName "w000003" ;
    sth:hasMachineId "cncA" ;
    sth:hasToolIdentifier "toolD6" ;
    sth:belongsToNCFile sth:NCFile_partA_op10_v3 .

sth:Window_w000001 sth:nextWindow sth:Window_w000002 .
sth:Window_w000002 sth:previousWindow sth:Window_w000001 ;
                 sth:nextWindow sth:Window_w000003 .
sth:Window_w000003 sth:previousWindow sth:Window_w000002 .

sth:DiagSeq_seq_partA_op10_001 rdf:type sth:DiagnosisSequence ;
    sth:hasSequenceIdentifier "seq_partA_op10_001" .

sth:PredSeq_seq_partA_op10_001 rdf:type sth:PredictionSequence ;
    sth:hasSequenceIdentifier "seq_partA_op10_001" .
"""


In [6]:

readme = f"""# CNC Starter Package (Updated with Sequence Extensions)

This starter package follows the workflow:

1. `examples/*.json`
2. `schema/*.json`
3. `ontology/sth.ttl`

It now includes both:
- atomic examples (single file/block/window/diagnosis/prediction)
- sequence examples (multi-window, diagnosis sequence, prediction sequence)

## Namespace alignment
- Root: `{ROOT}`
- Ontology namespace: `{ONTOLOGY_NS}`
- Schema base: `{SCHEMA_BASE}`
"""


In [ ]:

manifest = {
    "name": "cnc_starter_package_updated",
    "version": "1.1.0",
    "namespace_root": ROOT,
    "ontology": {"namespace": ONTOLOGY_NS, "local_file": "ontology/MC.ttl"},
    "examples": [{"name": k, "local_file": f"examples/{k}"} for k in examples.keys()],
    "schemas": [{"name": k, "id": v["$id"], "local_file": f"schema/{k}"} for k, v in schemas.items()]
}

for name, obj in examples.items():
    (examples_dir / name).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
for name, obj in schemas.items():
    (schema_dir / name).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
(ontology_dir / "MC.ttl").write_text(ttl_text, encoding="utf-8")
(root / "README.md").write_text(readme, encoding="utf-8")
(root / "starter_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

zip_path = base / "cnc_starter_package_updated.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in root.rglob("*"):
        if path.is_file():
            z.write(path, arcname=path.relative_to(base))

print("Created:")
print(root)
print(zip_path)
print("Examples:", len(list(examples_dir.glob('*.json'))))
print("Schemas:", len(list(schema_dir.glob('*.json'))))


Created:
.
cnc_starter_package_updated.zip
Examples: 11
Schemas: 9
